# Brecha de género en el capital humano y su relación con la capacidad de innovación empresarial

**Contexto del caso:**
- Empresas de los sectores servicios y manufactura reportan, año tras año, menos mujeres que hombres en puestos y roles ligados a innovación.
- La pregunta de este proyecto no es *si* existe la brecha, sino **qué tan fuerte pesa esa brecha frente a otros factores** (inversión, recursos propios, financiamiento) a la hora de explicar la capacidad de innovación de una empresa.
- Se trabaja con datos de la Encuesta de Desarrollo e Innovación Tecnológica (EDIT) del DANE, capítulos I a IV: tipos de innovación, inversión en ACTI, financiamiento y personal ocupado por género y nivel educativo.
- Artículo completo, con el diagrama del pipeline y la explicación paso a paso: **https://fuzzyfrog.ai/es/ai-lab/proyectos/negocio/brecha-genero-capital-humano-innovacion-empresarial-machine-learning/**

**Nota sobre los datos:** este notebook espera dos archivos `data/df_manufactura.csv` y `data/df_servicios.csv` con la estructura de códigos EDIT descrita más abajo. Los datos provienen de los microdatos anonimizados que publica el DANE en `https://microdatos.dane.gov.co/index.php/catalog/MICRODATOS` (catálogo EDIT, sector industria y sector servicios). Verifica la ronda exacta antes de descargar; el período de referencia se etiqueta aquí de forma genérica como **2025-2026**.


## Diagrama del pipeline

```
EDIT (DANE) ─┬─ df_manufactura.csv
             └─ df_servicios.csv
                     │
        Consolidación por código de pregunta (capítulos I-IV)
                     │
        Ingeniería de variables
        (capacidad_de_innovacion, diferencia_puestos,
         diferencia_educacion, equitativa, brecha_relativa)
                     │
        ┌────────────┼────────────┬───────────────┐
        │            │            │               │
   H1: XGBoost   H2: KMeans   H3: XGBoost    H4: Red neuronal
   Regresión     (perfiles)   Clasificación  + simulación de
   (capital                   (equitativa)   escenarios
   humano →
   innovación)
```

El diagrama interactivo (con tooltip por bloque) está embebido en el artículo de la plataforma.


## 1. Carga de datos

- Diccionario de códigos EDIT reducido a las variables que realmente alimentan el consolidado (capítulos I, II, III y IV).
- Se cargan por separado los CSV de manufactura y de servicios, y se filtran solo las columnas que existen en el diccionario.
- No se usa Google Drive: los CSV viven en `data/` dentro del repo.


In [ ]:
import pandas as pd

# --- Capítulo I: tipos de innovación e impactos ---
capitulo_1 = {
    "I1R1C1N": "Bienes o servicios nuevos únicamente para la empresa 2025",
    "I1R2C1N": "Bienes o servicios nuevos en el mercado nacional 2025",
    "I1R3C1N": "Bienes o servicios nuevos en el mercado internacional 2025",
    "I1R4C1": "Introdujo procesos nuevos o significativamente mejorados 2025",
    "I2R1C1": "Mejora en la calidad de los bienes o servicios",
    "I2R5C1": "Aumento de la productividad",
    "I2R6C1": "Reducción de los costos laborales",
    "I2R8C1": "Reducción en el consumo de energía eléctrica u otros energéticos",
    "I2R9C1": "Reducción en el consumo de agua",
    "I2R13C1": "Cumplimiento de regulaciones, normas y reglamentos técnicos",
    "I2R14C1": "Aprovechamiento de residuos en los procesos de la empresa",
    "I3R1C1": "Ventas nacionales totales 2025 (miles de pesos corrientes)",
    "I3R1C2": "Exportaciones totales 2025 (miles de pesos corrientes)",
}

# --- Capítulo II: inversión en ACTI ---
capitulo_2 = {
    "II1R1C1": "Monto invertido en I+D interna 2025",
    "II1R2C1": "Monto invertido en I+D externa 2025",
    "II1R3C1": "Monto invertido en maquinaria y equipo 2025",
    "II1R4C1": "Monto invertido en tecnologías de información y telecomunicaciones 2025",
    "II1R5C1": "Monto invertido en mercadotecnia 2025",
    "II1R10C1": "Monto total invertido en actividades de innovación 2025",
}

# --- Capítulo III: financiamiento ---
capitulo_3 = {
    "III1R1C1": "Recursos propios de la empresa 2025",
    "III1R3C1": "Recursos públicos para la realización de ACTI 2025",
    "III1R4C1": "Recursos de banca privada, nacional 2025",
    "III1R6C1": "Fondos de capital privado, nacional 2025",
    "III1R8C1": "Total de la inversión en ACTI 2025",
    "III4R1C1": "Barrera de financiamiento: desconocimiento de líneas públicas",
    "III5R1C1": "Obtuvo beneficios tributarios 2025-2026",
}

# --- Capítulo IV: personal ocupado, educación y género ---
capitulo_4 = {
    "IV1R11C1": "Total personal ocupado promedio 2025",
    "IV4R4C1": "Personal en producción, hombres, 2026",
    "IV4R4C2": "Personal en producción, mujeres, 2026",
    "IV4R11C1": "Total personal en ACTI, hombres, 2026",
    "IV4R11C2": "Total personal en ACTI, mujeres, 2026",
    "IV6R8C1": "Educación superior en ACTI, hombres, 2025",
    "IV6R8C2": "Educación superior en ACTI, mujeres, 2025",
}

capitulos = {
    "I": capitulo_1, "II": capitulo_2, "III": capitulo_3, "IV": capitulo_4,
}
global_codes = [code for cap in capitulos.values() for code in cap.keys()]
print(f"Total de códigos EDIT usados en el consolidado: {len(global_codes)}")


In [ ]:
# Rutas locales (el CSV se descarga de DANE y se sube al repo, no a Drive)
df_manufactura = pd.read_csv("data/df_manufactura.csv")
df_servicios = pd.read_csv("data/df_servicios.csv")

# Nos quedamos solo con las columnas del diccionario de códigos que sí existen en cada archivo
cols_manufactura = [c for c in global_codes if c in df_manufactura.columns]
cols_servicios = [c for c in global_codes if c in df_servicios.columns]

df_manufactura = df_manufactura[cols_manufactura]
df_servicios = df_servicios[cols_servicios]

print(f"Manufactura: {df_manufactura.shape[0]} empresas, {df_manufactura.shape[1]} variables")
print(f"Servicios: {df_servicios.shape[0]} empresas, {df_servicios.shape[1]} variables")

df = pd.concat([df_manufactura, df_servicios], ignore_index=True)
df = df.dropna(axis=1, how="all")


## 2. Explicación de los datos

- Cada fila es una empresa; cada columna, la respuesta a una pregunta específica del cuestionario EDIT.
- Los códigos siguen el patrón `CapítuloRfilaCcolumna` (ej. `II1R1C1` = capítulo II, pregunta 1, fila 1, columna 1).
- Antes de modelar, renombramos los códigos a nombres legibles y unificamos manufactura + servicios en un solo `df`.


In [ ]:
codigo_a_nombre = {
    "I3R1C1": "ventas_nacional", "I3R1C2": "ventas_exportacion",
    "I2R5C1": "aumento_productividad", "I2R6C1": "reduccion_costos",
    "I2R8C1": "reduccion_energia", "I2R9C1": "reduccion_agua",
    "I2R13C1": "cumplimiento_regulaciones", "I2R14C1": "aprovechamiento_residuos",
    "II1R10C1": "inversion_total",
    "III1R1C1": "Recursos propios", "III1R3C1": "Recursos externos",
    "III5R1C1": "Beneficios tributarios", "III4R1C1": "Barreras de financiamiento",
    "IV4R4C1": "puestos_hombres", "IV4R4C2": "puestos_mujeres",
    "IV6R8C1": "educacion_hombres", "IV6R8C2": "educacion_mujeres",
    "I1R1C1N": "innovacion_producto_servicio_a", "I1R2C1N": "innovacion_producto_servicio_b",
    "I1R3C1N": "innovacion_producto_servicio_c", "I1R4C1": "innovacion_proceso",
}
df = df.rename(columns=codigo_a_nombre)
df = df.fillna(0)
df.describe().T[["mean", "std", "min", "max"]]


## 3. Análisis exploratorio (EDA)

Antes de modelar cualquier hipótesis, dos preguntas simples: ¿cómo se ve la inversión en innovación en general?, y ¿cómo se ve la brecha de puestos y educación por género? Estas dos gráficas son la base de todo lo que sigue.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df["inversion_total"], bins=40, color="#006a87")
axes[0].set_title("Distribución de inversión total en innovación")
axes[0].set_xlabel("Inversión total (miles de pesos corrientes)")

puestos = df[["puestos_hombres", "puestos_mujeres"]].sum()
axes[1].bar(puestos.index, puestos.values, color=["#32ecfd", "#00b76c"])
axes[1].set_title("Puestos en ACTI por género (total muestra)")

plt.tight_layout()
plt.show()


## 4. Construcción de `capacidad_de_innovacion`

**Condición del proyecto:** se necesitaba una sola métrica objetivo que resumiera "cuánto innova" una empresa, a partir de variables de tipos de innovación (producto/servicio vs. proceso).

**Opción descartada:** un índice compuesto multi-factor (`III = α·I_tipo + β·I_inv + γ·I_fin + λ·ROI + μ·I_sust − δ·I_barreras`), inspirado en Manual de Oslo, Manual de Frascati y el European Innovation Scoreboard, con normalización 0-1 de cada componente. Se descartó porque exigía normalizar seis variables de escalas muy distintas con pesos arbitrarios (α, β, γ...) sin una base empírica clara para fijarlos, lo que habría inflado la varianza del indicador sin mejorar su interpretabilidad.

**Opción elegida:** una suma ponderada simple de dos señales directas de innovación de producto/servicio y de proceso, con más peso al producto/servicio porque es la señal con mayor cobertura de respuesta en la encuesta:


In [ ]:
df["innovacion_producto_servicio"] = (
    df[["innovacion_producto_servicio_a", "innovacion_producto_servicio_b", "innovacion_producto_servicio_c"]]
    .sum(axis=1)
)

peso_producto_servicio = 0.6
peso_proceso = 0.4

df["capacidad_de_innovacion"] = (
    df["innovacion_producto_servicio"] * peso_producto_servicio
    + df["innovacion_proceso"] * peso_proceso
)

df = df.drop(columns=[
    "innovacion_producto_servicio_a", "innovacion_producto_servicio_b",
    "innovacion_producto_servicio_c", "innovacion_producto_servicio", "innovacion_proceso",
])
df["capacidad_de_innovacion"].describe()


## 5. Modelado

### H1 — ¿Hay diferencias de capital humano por género que expliquen la capacidad de innovación?

Regresión con **XGBoost**: variable objetivo `capacidad_de_innovacion`, predictoras todo lo demás (inversión, financiamiento, puestos y educación por género). El criterio técnico aquí: se prefirió XGBoost sobre una regresión lineal porque las relaciones entre inversión, financiamiento y género no son lineales ni aditivas, y XGBoost captura interacciones sin tener que especificarlas a mano.


In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

X = df.drop(columns=["capacidad_de_innovacion"])
y = df["capacidad_de_innovacion"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

modelo_h1 = xgb.XGBRegressor(objective="reg:squarederror", random_state=42)
modelo_h1.fit(X_train_scaled, y_train)

y_pred = modelo_h1.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MSE: {mse:.3f} | R²: {r2:.3f}")

xgb.plot_importance(modelo_h1, max_num_features=10, importance_type="weight")
plt.title("Importancia de variables — H1")
plt.tight_layout()
plt.show()


**Proceso de iteración (resultado honesto):** el modelo consistentemente ubica `inversion_total` y `Recursos propios` por encima de cualquier variable de género. `puestos_hombres` sí aparece por encima de `puestos_mujeres` en importancia, lo que sostiene H1 (hay diferencia), pero la magnitud de esa diferencia es menor que la de los factores financieros — es decir, el dinero pesa más que el género para explicar cuánto innova una empresa, aunque el género no deja de pesar.


### H2 — ¿Existen perfiles de empresa distintos en innovación y género?

**KMeans** con 3 clústeres sobre las mismas variables (sin la variable objetivo). Se asumió independencia entre variables porque la encuesta no aporta evidencia de dependencia funcional entre ellas para efectos de segmentación.


In [ ]:
from sklearn.cluster import KMeans

variables_kmeans = [c for c in df.columns if c != "capacidad_de_innovacion"]
X_kmeans = StandardScaler().fit_transform(df[variables_kmeans])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["perfil"] = kmeans.fit_predict(X_kmeans)

resumen_perfiles = df.groupby("perfil")[
    ["inversion_total", "capacidad_de_innovacion", "puestos_mujeres", "puestos_hombres"]
].mean()
resumen_perfiles


**Lectura de los perfiles:** el perfil con mayor inversión también concentra la mayor cantidad de puestos ocupados por mujeres en términos absolutos — no porque haya "más igualdad" en esas empresas, sino porque son empresas más grandes con más puestos en general. Esta distinción importa: el tamaño de la empresa confunde la lectura de equidad si no se normaliza, algo que se corrige en H3.


### H3 — ¿Las barreras estructurales limitan la equidad de género en innovación?

Aquí se construyen dos variables nuevas para no depender de conteos absolutos (que favorecen a empresas grandes): la diferencia entre puestos y entre educación por género, y a partir de ellas una variable binaria `equitativa`.


In [ ]:
df["diferencia_puestos"] = (df["puestos_mujeres"] - df["puestos_hombres"]).abs()
df["diferencia_educacion"] = (df["educacion_mujeres"] - df["educacion_hombres"]).abs()

df["equitativa"] = (
    (df["diferencia_puestos"] < 2) & (df["diferencia_educacion"] < 2)
).astype(int)

print(df["equitativa"].value_counts(normalize=True))


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

predictoras_h3 = [
    "Barreras de financiamiento", "Recursos propios", "Recursos externos",
    "ventas_nacional", "ventas_exportacion", "aumento_productividad", "reduccion_costos",
]
X3 = df[predictoras_h3]
y3 = df["equitativa"]

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42, stratify=y3)

clf_h3 = xgb.XGBClassifier(objective="binary:logistic", random_state=42, eval_metric="logloss")
clf_h3.fit(X3_train, y3_train)

y3_pred = clf_h3.predict(X3_test)
print(f"Accuracy: {accuracy_score(y3_test, y3_pred):.3f}")
print(classification_report(y3_test, y3_pred))

xgb.plot_importance(clf_h3, importance_type="weight")
plt.title("Importancia de variables — H3 (equidad)")
plt.tight_layout()
plt.show()


**Proceso de iteración (resultado honesto):** `Recursos propios` domina la importancia de variables muy por encima de `Barreras de financiamiento`. Esto es una limitación real del análisis: la variable de barreras reportada en la encuesta tiene muy poca variación (la enorme mayoría responde "sin barreras"), así que el modelo termina apoyándose en recursos internos como proxy, no en la barrera estructural en sí. No se fuerza una lectura más favorable: el dato de barreras, tal como está reportado, aporta poca señal.


### H4 — Modelo de gestión: simular el efecto de más equidad de género

Una red neuronal (MLP) entrenada sobre `capacidad_de_innovacion`, con una variable adicional `brecha_relativa` (brecha de género normalizada). El objetivo no es solo predecir, sino **simular escenarios**: una vez que el modelo aprende cómo se comporta la capacidad de innovación, se le presentan casos hipotéticos donde cambia la participación de mujeres, para ver el efecto marginal.


In [ ]:
from sklearn.model_selection import train_test_split as tts
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df["brecha_relativa"] = (
    (df["puestos_hombres"] - df["puestos_mujeres"])
    / (df["puestos_hombres"] + df["puestos_mujeres"] + 1e-6)
)

predictoras_h4 = [
    "inversion_total", "Recursos propios", "Recursos externos", "Beneficios tributarios",
    "Barreras de financiamiento", "ventas_nacional", "ventas_exportacion",
    "aumento_productividad", "reduccion_costos", "reduccion_energia", "reduccion_agua",
    "puestos_mujeres", "educacion_mujeres", "puestos_hombres", "educacion_hombres",
    "brecha_relativa",
]
X4 = df[predictoras_h4]
y4 = df["capacidad_de_innovacion"]

scaler_h4 = StandardScaler()
X4_scaled = scaler_h4.fit_transform(X4)

X4_train, X4_test, y4_train, y4_test = tts(X4_scaled, y4, test_size=0.2, random_state=42)

modelo_h4 = Sequential([
    Dense(64, input_dim=X4_train.shape[1], activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="linear"),
])
modelo_h4.compile(loss="mean_squared_error", optimizer="adam")
modelo_h4.fit(X4_train, y4_train, epochs=100, batch_size=32, validation_split=0.2, verbose=0)

perdida = modelo_h4.evaluate(X4_test, y4_test, verbose=0)
print(f"Pérdida (MSE) en prueba: {perdida:.3f}")


In [ ]:
import numpy as np

# Escenario base: un caso real del set de prueba
caso_base = X4_test[0].copy()
prediccion_base = modelo_h4.predict(np.array([caso_base]), verbose=0)[0][0]

# Mismo caso, con más mujeres en puestos y en educación (índices según predictoras_h4)
idx_puestos_mujeres = predictoras_h4.index("puestos_mujeres")
idx_educacion_mujeres = predictoras_h4.index("educacion_mujeres")

caso_politica = caso_base.copy()
caso_politica[idx_puestos_mujeres] = 50
caso_politica[idx_educacion_mujeres] = 20
prediccion_politica = modelo_h4.predict(np.array([caso_politica]), verbose=0)[0][0]

cambio_pct = (prediccion_politica - prediccion_base) / abs(prediccion_base) * 100
print(f"Predicción base:            {prediccion_base:.4f}")
print(f"Predicción con más equidad: {prediccion_politica:.4f}")
print(f"Cambio: {cambio_pct:+.1f}%")


**Validación H4:** en el escenario simulado, aumentar puestos y educación de mujeres eleva la predicción de capacidad de innovación frente al caso base — el modelo de gestión que sugiere el proyecto no es "contratar más mujeres porque sí", sino que la inclusión de género en el capital humano se comporta, dentro de los límites de este modelo, como una palanca más de innovación, no como un costo aislado de las demás. Esto no reemplaza los factores financieros identificados en H1 y H3, los complementa.


## 6. Evaluación

| Hipótesis | Modelo | Métrica | Resultado |
|---|---|---|---|
| H1 | XGBoost Regresión | R² | ver celda de H1 |
| H2 | KMeans (k=3) | Perfiles diferenciados | ver tabla de perfiles |
| H3 | XGBoost Clasificación | Accuracy | ver celda de H3 |
| H4 | Red neuronal (MLP) | MSE + simulación de escenario | ver celdas de H4 |


## 7. Hallazgos principales

- Los factores financieros internos (inversión total, recursos propios) explican más la capacidad de innovación que cualquier variable de género tomada sola — pero el género no deja de tener peso propio.
- Los perfiles de empresa muestran que el tamaño confunde la lectura de equidad: más puestos para mujeres en términos absolutos no siempre significa más equidad relativa.
- La variable de "barreras de financiamiento" reportada en la encuesta tiene poca variación, lo que limita qué tanto puede decir un modelo sobre barreras estructurales con estos datos específicos.
- La simulación de escenarios (H4) sugiere que aumentar la participación de mujeres en puestos y educación empuja la predicción de capacidad de innovación hacia arriba, dentro de los límites del modelo entrenado — evidencia de que la inclusión de género acompaña la innovación, no la compite.
